<a href="https://colab.research.google.com/github/mjrboi-MicahR/ScamML/blob/main/MLSpam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import pandas as pd
import numpy as np
import kagglehub
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, GlobalAveragePooling1D, TextVectorization

# First data set
print("Downloading first dataset (spam-email-dataset)...")
path1 = kagglehub.dataset_download("jackksoncsie/spam-email-dataset")
csv_files1 = [f for f in os.listdir(path1) if f.endswith('.csv')]
csv_path1 = os.path.join(path1, csv_files1[0])
df1 = pd.read_csv(csv_path1)

# Standardize column names
df1 = df1.rename(columns={'text': 'text', 'spam': 'spam'})
df1 = df1[['text', 'spam']]
print(f"Dataset 1 shape: {df1.shape}")

# Second data set
print("\nDownloading second dataset (email-spam-classification-dataset)...")
path2 = kagglehub.dataset_download("purusinghvi/email-spam-classification-dataset")
csv_files2 = [f for f in os.listdir(path2) if f.endswith('.csv')]
csv_path2 = os.path.join(path2, csv_files2[0])
df2 = pd.read_csv(csv_path2)

# Identify columns dynamically
text_col_df2 = 'email' if 'email' in df2.columns else df2.columns[1]
label_col_df2 = 'label' if 'label' in df2.columns else df2.columns[0]

df2_cleaned = pd.DataFrame()
df2_cleaned['text'] = df2[text_col_df2].astype(str)

# Standardize string labels ('spam' / 'ham') to binary numeric values
if df2[label_col_df2].dtype == object:
    df2_cleaned['spam'] = df2[label_col_df2].str.lower().map({'spam': 1, 'ham': 0})
else:
    df2_cleaned['spam'] = df2[label_col_df2].astype(int)

df2_cleaned = df2_cleaned.dropna().reset_index(drop=True)
print(f"Dataset 2 standardized shape: {df2_cleaned.shape}")

# merge the data sets, then shuffle them, then split them into training and two testing sets
combined_df = pd.concat([df1, df2_cleaned], ignore_index=True)
combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)
print(f"\nCombined Dataset shape: {combined_df.shape}")
print(combined_df['spam'].value_counts())

X_comb = combined_df['text'].astype(str).values
y_comb = combined_df['spam'].values

# 70% Train, 15% Validation, 15% Test
X_train, X_temp, y_train, y_temp = train_test_split(
    X_comb, y_comb, test_size=0.30, random_state=42, stratify=y_comb
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"\nFinal Splits:")
print(f"- Train size: {len(X_train)}")
print(f"- Validation (Preliminary) size: {len(X_val)}")
print(f"- Testing size: {len(X_test)}")

# Model training
max_features = 12000
sequence_length = 250

vectorize_layer = TextVectorization(
    max_tokens=max_features,
    output_mode='int',
    output_sequence_length=sequence_length
)

# Train vectorize layer on the training set
vectorize_layer.adapt(X_train)

model = Sequential([
    vectorize_layer,
    Embedding(max_features, 64),
    GlobalAveragePooling1D(),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("\nTraining the model on the merged dataset...")
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_val, y_val)
)

# Testing the sets
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"\n--- Model Performance on Final Testing Set ---")
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy*100:.2f}%")

Using Colab cache for faster access to the 'spam-email-dataset' dataset.
Dataset 1 shape: (5728, 2)

Using Colab cache for faster access to the 'email-spam-classification-dataset' dataset.
Dataset 2 standardized shape: (83448, 2)

Combined Dataset shape: (89176, 2)
spam
1    45278
0    43898
Name: count, dtype: int64

Final Splits:
- Train size: 62423
- Validation (Preliminary) size: 13376
- Testing size: 13377

Training the model on the merged dataset...
Epoch 1/5
1951/1951 ━━━━━━━━━━━━━━━━━━━━ 38s 19ms/step - accuracy: 0.9386 - loss: 0.1612 - val_accuracy: 0.9769 - val_loss: 0.0718
Epoch 2/5
1951/1951 ━━━━━━━━━━━━━━━━━━━━ 40s 20ms/step - accuracy: 0.9778 - loss: 0.0638 - val_accuracy: 0.9692 - val_loss: 0.0823
Epoch 3/5
1951/1951 ━━━━━━━━━━━━━━━━━━━━ 40s 20ms/step - accuracy: 0.9827 - loss: 0.0496 - val_accuracy: 0.9599 - val_loss: 0.1221
Epoch 4/5
1951/1951 ━━━━━━━━━━━━━━━━━━━━ 37s 19ms/step - accuracy: 0.9846 - loss: 0.0430 - val_accuracy: 0.9697 - val_loss: 0.0775
Epoch 5/5
1951/1

In [ ]:
### Merged Training Pipeline
This section consolidates dataset downloading, parsing, preparation, model architecture, training, and evaluation into a single execution pass.

Using Colab cache for faster access to the 'spam-email-dataset' dataset.
Dataset 1 shape: (5728, 2)



100%|██████████| 43.0M/43.0M [00:03<00:00, 13.3MB/s]

Extracting files...


Dataset 2 standardized shape: (83448, 2)

Combined Dataset shape: (89176, 2)
spam
1    45278
0    43898
Name: count, dtype: int64

Final Splits:
- Train size: 62423
- Validation (Preliminary) size: 13376
- Testing size: 13377


In [ ]:
---

_IncompleteInputError: incomplete input (330595976.py, line 45)

In [ ]:
#@title Interactive Spam Classifier
# Run this cell to test custom email transcripts against the trained Keras model.

import numpy as np

def predict_email(email_text):
    # Convert the raw text into an object array so Keras can process the string correctly
    input_data = np.array([email_text], dtype=object)
    prediction = model.predict(input_data, verbose=0)[0][0]

    print("\n" + "="*40)
    print("RESULTS:")
    print("="*40)
    if prediction >= 0.5:
        print(f"Prediction: SPAM (Confidence: {prediction * 100:.2f}%)")
    else:
        print(f"Prediction: NOT SPAM (Confidence: {(1 - prediction) * 100:.2f}%)")
    print("="*40)

# Prompt user for input
user_email = input("Paste the email transcript here to analyze: ")
if user_email.strip():
    predict_email(user_email)
else:
    print("Please provide a valid text input next time!")

Paste the email transcript here to analyze: Subject: Updated Document Shared with You: Q4 Project Schedule & Budget Review  Hi there, A user has shared a document with you on Google Drive: "Q4_Project_Schedule_&_Budget_Review.pdf". You can review the updated timelines, resource allocations, and comments attached to the document by clicking the link below: 👉 Open Document in Workspace Note: This link will expire in 7 days for security purposes. If you did not expect this document, you can manage your notification settings in your profile. Best regards, Cloud Storage Notifications Automation Team

RESULTS:
Prediction: NOT SPAM (Confidence: 99.87%)
